# YogaPose StyleGAN2-ADA Colab (A100-ready)

This notebook trains a **StyleGAN2-ADA** model on a **subset of 5–10 yoga pose classes** and then generates + displays synthetic samples.

## Key reliability fixes in this version
- Downloads dataset via **Kaggle CLI** (same method as the rest of this project)
- Extracts to `/content/local_data/computer-vision/yogaposes` to match project path conventions
- Prints dataset discovery + class folder names
- Writes **all outputs** under a single Drive root
- Adds explicit checks for: training zip contents, run dirs, snapshots, generated images
- Makes the `generate.py` call use quoted paths to avoid issues with spaces

If generation shows **0 images**, the diagnostic cells will tell you whether: training never produced snapshots, the wrong snapshot is selected, or output is going to a different folder.

---
## Cell 1 — Mount Drive and configure paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# ── Local dataset path (matches the rest of this project) ──────────────────────
local_base_parent = '/content/local_data/computer-vision/yogaposes'
local_base        = os.path.join(local_base_parent, '107 yoga poses')
DATASET_TRAIN_DIR = os.path.join(local_base, 'train')   # class subfolders live here

# ── Drive root for all StyleGAN outputs ────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/YogaPoseStyleGAN'
SUBSET_DIR    = os.path.join(DRIVE_ROOT, 'subset_dataset')
STYLEGAN_REPO = '/content/stylegan2-ada-pytorch'
TRAIN_ZIP     = os.path.join(DRIVE_ROOT, 'yoga_subset_stylegan.zip')
RUNS_DIR      = os.path.join(DRIVE_ROOT, 'training-runs')
GENERATED_DIR = os.path.join(DRIVE_ROOT, 'generated_samples')

for d in [local_base_parent, DRIVE_ROOT, SUBSET_DIR, RUNS_DIR, GENERATED_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted ✓')
print('local_base        =', local_base)
print('DATASET_TRAIN_DIR =', DATASET_TRAIN_DIR)
print('DRIVE_ROOT        =', DRIVE_ROOT)
print('SUBSET_DIR        =', SUBSET_DIR)
print('TRAIN_ZIP         =', TRAIN_ZIP)
print('RUNS_DIR          =', RUNS_DIR)
print('GENERATED_DIR     =', GENERATED_DIR)

---
## Cell 2 — Install dependencies

In [ ]:
!pip install -q pillow tqdm matplotlib kaggle

---
## Cell 3 — Download and extract dataset via Kaggle CLI

Uses the same download method as the rest of this project.  
Make sure your Kaggle API token (`kaggle.json`) is uploaded to Colab or stored at `~/.kaggle/kaggle.json` before running.

In [ ]:
import os

local_base_parent = '/content/local_data/computer-vision/yogaposes'
local_base        = os.path.join(local_base_parent, '107 yoga poses')
DATASET_TRAIN_DIR = os.path.join(local_base, 'train')

if not os.path.exists(DATASET_TRAIN_DIR):
    print('Downloading dataset from Kaggle...')
    !kaggle datasets download -d arrowe/yoga-poses-dataset-107

    print(f'Unzipping dataset to {local_base_parent}...')
    !unzip -o "/content/yoga-poses-dataset-107.zip" -d {local_base_parent}

    # Remove zip to save space
    !rm "/content/yoga-poses-dataset-107.zip"
    print('Dataset ready ✓')
else:
    print('Dataset already extracted ✓')

# Verify
print('\nContents of local_base_parent:')
!ls -la {local_base_parent}
print('\nContents of local_base (107 yoga poses):')
!ls -la "{local_base}"

---
## Cell 4 — Discover class folders in train split

In [ ]:
import os

DATASET_TRAIN_DIR = '/content/local_data/computer-vision/yogaposes/107 yoga poses/train'

if not os.path.isdir(DATASET_TRAIN_DIR):
    raise RuntimeError(f'Train directory not found: {DATASET_TRAIN_DIR}\nRe-run Cell 3 to download and extract the dataset.')

classes = sorted([d for d in os.listdir(DATASET_TRAIN_DIR)
                  if os.path.isdir(os.path.join(DATASET_TRAIN_DIR, d))])

print(f'DATASET_TRAIN_DIR = {DATASET_TRAIN_DIR}')
print(f'Total class folders found: {len(classes)}')
print(f'First 30 class folder names: {classes[:30]}')

---
## Cell 5 — Choose 5–10 pose classes + image size

In [ ]:
# ⚙️ Optionally override with specific class names printed in Cell 4
# e.g. SELECTED_CLASSES = ['adho mukha svanasana', 'bakasana', 'vriksasana']
SELECTED_CLASSES = []

# Default: pick first 8 classes alphabetically if none specified
if not SELECTED_CLASSES:
    SELECTED_CLASSES = classes[:8]

IMG_SIZE             = 256   # StyleGAN2-ADA works well at 256x256
MAX_IMAGES_PER_CLASS = 300   # cap per class to keep training set balanced

print('Selected classes     :', SELECTED_CLASSES)
print('IMG_SIZE             :', IMG_SIZE)
print('MAX_IMAGES_PER_CLASS :', MAX_IMAGES_PER_CLASS)

---
## Cell 6 — Build subset dataset (resize + pad to square)

In [ ]:
import shutil
from PIL import Image, ImageOps
from tqdm import tqdm

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# Clear and recreate subset dir
if os.path.exists(SUBSET_DIR):
    shutil.rmtree(SUBSET_DIR)
os.makedirs(SUBSET_DIR, exist_ok=True)

def resize_and_pad(img, size):
    """Resize preserving aspect ratio, pad with white to exact square."""
    img = ImageOps.exif_transpose(img).convert('RGB')
    img.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), (255, 255, 255))
    x = (size - img.width)  // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

summary = {}
for cls in SELECTED_CLASSES:
    src_dir = os.path.join(DATASET_TRAIN_DIR, cls)
    dst_dir = os.path.join(SUBSET_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        print(f'WARNING: class folder not found: {cls} (check spelling against Cell 4 output)')
        summary[cls] = 0
        continue

    files = [f for f in sorted(os.listdir(src_dir))
             if os.path.splitext(f)[1].lower() in VALID_EXTS]
    files = files[:MAX_IMAGES_PER_CLASS]

    count = 0
    for fname in tqdm(files, desc=cls):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dst_dir, os.path.splitext(fname)[0] + '.png')
        try:
            out = resize_and_pad(Image.open(src), IMG_SIZE)
            out.save(dst, format='PNG')
            count += 1
        except Exception:
            pass

    summary[cls] = count

total = sum(summary.values())
print('\nSubset build complete ✓')
print(f'Total images in subset: {total}')
for k, v in summary.items():
    print(f'  {k:40s} {v}')

if total == 0:
    raise RuntimeError(
        'Subset contains 0 images. '
        'Check that SELECTED_CLASSES matches folder names printed in Cell 4.')

---
## Cell 7 — Create training ZIP and verify contents

In [ ]:
import zipfile

if os.path.exists(TRAIN_ZIP):
    os.remove(TRAIN_ZIP)

with zipfile.ZipFile(TRAIN_ZIP, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, _, files in os.walk(SUBSET_DIR):
        for f in files:
            full = os.path.join(root, f)
            rel  = os.path.relpath(full, SUBSET_DIR)
            zf.write(full, rel)

with zipfile.ZipFile(TRAIN_ZIP, 'r') as zf:
    names = zf.namelist()

print('Training zip created ✓')
print('Path       :', TRAIN_ZIP)
print('Files in zip:', len(names))
print('First 10   :', names[:10])

if len(names) == 0:
    raise RuntimeError('Training zip is empty. Re-run Cell 6 and check subset images were created.')

---
## Cell 8 — Clone StyleGAN2-ADA and install requirements

In [ ]:
if not os.path.exists(STYLEGAN_REPO):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git "{STYLEGAN_REPO}"
else:
    print('StyleGAN2-ADA repo already cloned ✓')

%cd "{STYLEGAN_REPO}"
!pip install -q -r requirements.txt

---
## Cell 9 — Training settings

| Setting | Value | Notes |
|---|---|---|
| `KIMG` | 200 | Total training images (thousands). Increase for better quality. |
| `SNAP` | 20 | Save a snapshot every N kimg. |
| `BATCH` | 16 | Reduce to 8 if GPU OOM. |
| `AUG` | ada | Adaptive discriminator augmentation — essential for small datasets. |

In [ ]:
!nvidia-smi

KIMG  = 200
SNAP  = 20
BATCH = 16
AUG   = 'ada'

print(f'KIMG={KIMG}  SNAP={SNAP}  BATCH={BATCH}  AUG={AUG}')

---
## Cell 10 — Launch training (Drive-backed outdir)

In [ ]:
!python train.py \
  --outdir="{RUNS_DIR}" \
  --data="{TRAIN_ZIP}" \
  --gpus=1 \
  --cfg=auto \
  --snap={SNAP} \
  --kimg={KIMG} \
  --batch={BATCH} \
  --aug={AUG}

---
## Cell 11 — Find latest snapshot + diagnose runs

In [ ]:
import glob

run_dirs = sorted(glob.glob(os.path.join(RUNS_DIR, '*')))
print('Run dirs found :', len(run_dirs))
print('Last 3 run dirs:', run_dirs[-3:])

latest_run = run_dirs[-1] if run_dirs else None
print('Latest run     :', latest_run)

snapshots = sorted(
    glob.glob(os.path.join(latest_run, 'network-snapshot-*.pkl'))
) if latest_run else []

print('Snapshots found:', len(snapshots))
NETWORK_PKL = snapshots[-1] if snapshots else None
print('Latest snapshot:', NETWORK_PKL)

if NETWORK_PKL is None:
    raise RuntimeError(
        'No network snapshot found. Training may have failed or SNAP/KIMG '
        'settings prevented any snapshots being saved. Inspect Cell 10 output.')

---
## Cell 12 — Generate images and verify output

In [ ]:
import glob

# Clear any previous generated images
for f in glob.glob(os.path.join(GENERATED_DIR, '*.png')):
    os.remove(f)

!python generate.py \
  --outdir="{GENERATED_DIR}" \
  --trunc=0.8 \
  --seeds=0-24 \
  --network="{NETWORK_PKL}"

generated = sorted(glob.glob(os.path.join(GENERATED_DIR, '*.png')))
print(f'Generated : {len(generated)} png files')
print(f'Output dir: {GENERATED_DIR}')

if len(generated) == 0:
    raise RuntimeError(
        'generate.py produced 0 images. '
        'Check the output above for errors and confirm NETWORK_PKL is a valid .pkl file.')

---
## Cell 13 — Display generated images inline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

sample_files = sorted(glob.glob(os.path.join(GENERATED_DIR, '*.png')))
print(f'Found {len(sample_files)} generated images in {GENERATED_DIR}')

n    = min(25, len(sample_files))
grid = 5

fig, axes = plt.subplots(grid, grid, figsize=(12, 12))
axes = axes.flatten()

for i in range(grid * grid):
    ax = axes[i]
    if i < n:
        ax.imshow(mpimg.imread(sample_files[i]))
        ax.set_title(os.path.basename(sample_files[i]), fontsize=7)
    ax.axis('off')

plt.suptitle('Generated YogaPose Samples (StyleGAN2-ADA)', fontsize=14)
plt.tight_layout()
plt.show()